# LangSmith CI/CD Evaluation Demo

This notebook demonstrates a complete CI/CD pipeline that automatically evaluates your LangGraph agent whenever code changes are made.

## Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Code Change    │────▶│  GitHub Action   │────▶│  LangSmith      │
│  (graph.py)     │     │  Triggers Eval   │     │  Evaluation     │
└─────────────────┘     └──────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
                        ┌──────────────────┐     ┌─────────────────-┐
                        │  PR Comment      │◀────│  Score Report    │
                        │  PASS / FAIL     │     │  (threshold 0.7) │
                        └──────────────────┘     └────────────────-─┘
```

**What you'll see:**
1. Multi-agent system implementation
2. Database tools for the agents
3. LangSmith evaluation setup
4. GitHub Actions workflow
5. Live demo: trigger an eval by opening a PR

---

# Part 1: The Multi-Agent System

Our agent uses a **supervisor pattern** with two specialized sub-agents:
- **Invoice Agent**: Handles billing and purchase queries
- **Music Agent**: Handles music catalog queries

The supervisor routes incoming questions to the appropriate agent.

## Agent Implementation (`app/graph.py`)

```python
from typing import Annotated
from typing_extensions import TypedDict, NotRequired

from langchain_openai import ChatOpenAI
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

from app.tools import invoice_tools, music_tools


# State schema
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    remaining_steps: NotRequired[RemainingSteps]


# Model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Invoice Subagent
invoice_prompt = """You are an invoice specialist for a digital music store.

Tools available:
- get_invoices_by_customer: Look up invoices by customer ID
- get_invoice_total: Get total spending for a customer

CRITICAL: Your response MUST include ALL specific data retrieved:
- For invoices: List each Invoice ID, date, and total amount
- For totals: State the exact dollar amount"""

invoice_agent = create_react_agent(
    model,
    tools=invoice_tools,
    name="invoice_agent",
    prompt=invoice_prompt,
    state_schema=State,
)


# Music Catalog Subagent
music_prompt = """You are a music catalog specialist for a digital music store.

Tools available:
- get_albums_by_artist: Search albums by artist name
- get_tracks_by_artist: Get songs/tracks by artist
- search_tracks: Search tracks by name

CRITICAL: Your response MUST include ALL specific data retrieved:
- List actual album titles, track names, and artist names
- Do not summarize or abbreviate results"""

music_agent = create_react_agent(
    model,
    tools=music_tools,
    name="music_agent",
    prompt=music_prompt,
    state_schema=State,
)


# Supervisor - routes to appropriate agent
supervisor_prompt = """You are a customer support supervisor for a digital music store.

IMMEDIATELY route customer inquiries - do NOT ask permission, just transfer:
1. **invoice_agent**: ANY question about billing, invoices, purchases, spending
2. **music_agent**: ANY question about music, albums, artists, tracks, songs"""

workflow = create_supervisor(
    agents=[invoice_agent, music_agent],
    model=model,
    prompt=supervisor_prompt,
    state_schema=State,
    output_mode="full_history",
)

graph = workflow.compile()


async def run_graph(inputs: dict) -> dict:
    """Run the multi-agent graph and return the final response."""
    result = await graph.ainvoke(inputs)
    final_message = result["messages"][-1]
    return {"output": final_message.content}
```

## Agent Tools (`app/tools.py`)

The agents use tools that query the **Chinook sample database** (a music store database with customers, invoices, albums, and tracks).

```python
import sqlite3
import requests
from langchain_core.tools import tool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db():
    """Load Chinook sample database into memory."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)


# Invoice Tools
@tool
def get_invoices_by_customer(customer_id: str) -> str:
    """Get all invoices for a customer, sorted by date (most recent first)."""
    return db.run(
        f"SELECT * FROM Invoice WHERE CustomerId = {customer_id} ORDER BY InvoiceDate DESC LIMIT 5;"
    )


@tool
def get_invoice_total(customer_id: str) -> str:
    """Get the total amount spent by a customer across all invoices."""
    return db.run(
        f"SELECT SUM(Total) as TotalSpent FROM Invoice WHERE CustomerId = {customer_id};"
    )


invoice_tools = [get_invoices_by_customer, get_invoice_total]


# Music Catalog Tools
@tool
def get_albums_by_artist(artist: str) -> str:
    """Search for albums by an artist name."""
    return db.run(
        f"""
        SELECT Album.Title, Artist.Name
        FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE '%{artist}%'
        LIMIT 10;
        """,
        include_columns=True,
    )


@tool
def get_tracks_by_artist(artist: str) -> str:
    """Get songs/tracks by an artist."""
    return db.run(
        f"""
        SELECT Track.Name as Song, Album.Title as Album, Artist.Name as Artist
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE '%{artist}%'
        LIMIT 10;
        """,
        include_columns=True,
    )


@tool
def search_tracks(query: str) -> str:
    """Search for tracks by name."""
    return db.run(
        f"""
        SELECT Track.Name as Song, Artist.Name as Artist, Album.Title as Album
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Track.Name LIKE '%{query}%'
        LIMIT 10;
        """,
        include_columns=True,
    )


music_tools = [get_albums_by_artist, get_tracks_by_artist, search_tracks]
```

In [1]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from app import run_graph

# Test query
result = await run_graph({
    "messages": [{"role": "user", "content": "What albums does AC/DC have?"}]
})

print("Response:", result["output"])

Task supervisor with path ('__pregel_pull', 'supervisor') wrote to unknown channel remaining_steps, ignoring it.


Response: AC/DC has the following albums:
1. **For Those About To Rock We Salute You**
2. **Let There Be Rock**


---

# Part 2: The Evaluation System

The evaluation uses:
- **LangSmith Dataset**: Test cases with expected answers
- **LLM-as-Judge**: GPT-4o-mini evaluates semantic correctness
- **Threshold**: 70% score required to pass

## Evaluation Test (`tests/test_eval.py`)

```python
import json
import pytest
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langsmith import Client
from openevals.llm import create_async_llm_as_judge

from app.graph import run_graph

# Initialize LangSmith client
client = Client()

# Custom prompt for semantic correctness (less strict on wording)
SEMANTIC_CORRECTNESS_PROMPT = """You are evaluating whether an AI response contains the key facts from the expected answer.

<User Question>
{inputs}
</User Question>

<AI Response>
{outputs}
</AI Response>

<Expected Answer>
{reference_outputs}
</Expected Answer>

Evaluate based on SEMANTIC CORRECTNESS:
- Does the AI response contain the same KEY FACTS as the expected answer?
- Minor wording differences are OK
- Additional helpful information beyond expected is OK
- The response should NOT contradict or omit key facts from expected

Score 1 if the response contains all key facts from expected.
Score 0 if key facts are missing, incorrect, or contradicted."""

# Create LLM-as-judge evaluator
judge_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
correctness_evaluator = create_async_llm_as_judge(
    prompt=SEMANTIC_CORRECTNESS_PROMPT,
    feedback_key="correctness",
    judge=judge_model,
)


@pytest.mark.evaluator
@pytest.mark.asyncio
async def test_multiagent_evaluation():
    """
    Run evaluation against a LangSmith dataset.
    
    Dataset format:
    - Input: {"messages": [{"role": "user", "content": "your question"}]}
    - Output: {"output": "expected answer"}
    """
    DATASET_NAME = "CICD Standalone: Multi-Agent Eval"
    EXPERIMENT_PREFIX = "multiagent-eval"
    PASSING_THRESHOLD = 0.7

    # Run evaluation
    experiment_results = await client.aevaluate(
        run_graph,
        data=DATASET_NAME,
        evaluators=[correctness_evaluator],
        experiment_prefix=EXPERIMENT_PREFIX,
        num_repetitions=1,
        max_concurrency=5,
    )

    # Save config for report generation
    criteria = {"correctness": f">={PASSING_THRESHOLD}"}
    output_metadata = {
        "experiment_name": experiment_results.experiment_name,
        "criteria": criteria,
    }

    safe_name = experiment_results.experiment_name.replace(":", "-").replace("/", "-")
    with open(f"evaluation_config__{safe_name}.json", "w") as f:
        json.dump(output_metadata, f, indent=2)
```

---

# Part 3: GitHub Actions Workflow

The workflow:
1. **Triggers** on PRs that modify `app/graph.py` or `app/tools.py`
2. **Runs** the evaluation test via pytest
3. **Posts** results as a PR comment

---

# Part 4: Trigger an Eval (Live Demo)

Now let's trigger an actual evaluation by:
1. Making a code change to the agent
2. Creating a branch and PR
3. Watching the GitHub Action run and post results

**Prerequisites:**
- `gh` CLI installed and authenticated (`gh auth login`)
- Git configured with push access to the repo

In [ ]:
import subprocess
import uuid
import re
from datetime import datetime

def run(cmd: str) -> str:
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="..")
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        raise Exception(result.stderr)
    return result.stdout.strip()

# Configuration
BRANCH_NAME = f"eval-test-{uuid.uuid4().hex[:8]}"
AGENT_FILE = "app/graph.py"

print(f"Branch name: {BRANCH_NAME}")

In [5]:
# Step 1: Ensure we're on main and up to date
print(run("git checkout main"))
print(run("git pull origin main"))

NameError: name 'run' is not defined

In [10]:
# Step 2: Create a new branch
print(run(f"git checkout -b {BRANCH_NAME}"))

In [11]:
# Step 3: Update the version string in the agent file
with open(f"../{AGENT_FILE}", "r") as f:
    content = f.read()

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
new_version = f"Version: notebook-triggered @ {timestamp}"

updated_content = re.sub(r'Version:.*', new_version, content)

with open(f"../{AGENT_FILE}", "w") as f:
    f.write(updated_content)

print(f"Updated {AGENT_FILE} with: {new_version}")

Updated app/graph.py with: Version: notebook-triggered @ 2026-02-18 22:13


In [12]:
# Step 4: Commit the change
run(f"git add {AGENT_FILE}")
print(run('git commit -m "test: trigger eval from notebook"'))

[eval-test-045e91c0 a87538a] test: trigger eval from notebook
 1 file changed, 1 insertion(+), 1 deletion(-)


In [13]:
# Step 5: Push the branch
print(run(f"git push -u origin {BRANCH_NAME}"))

branch 'eval-test-045e91c0' set up to track 'origin/eval-test-045e91c0'.


In [14]:
# Step 6: Create a PR
pr_title = "test: notebook-triggered eval"
pr_body = f"""## Summary
Automated PR created from Jupyter notebook to trigger eval workflow.

**Timestamp:** {timestamp}
**Branch:** {BRANCH_NAME}
"""

pr_url = run(f'gh pr create --title "{pr_title}" --body "{pr_body}"')
print(f"\n{'='*50}")
print(f"PR Created: {pr_url}")
print(f"{'='*50}")


PR Created: https://github.com/xuro-langchain/evals-cicd/pull/3


In [15]:
# Step 7: Open PR in browser to watch the workflow
print("Opening PR in browser...")
run("gh pr view --web")

Opening PR in browser...


''

---

## Cleanup

After watching the eval complete, run these cells to clean up.

In [ ]:
# Close the PR and delete the branch
print(run(f"gh pr close {BRANCH_NAME} --delete-branch"))

In [ ]:
# Switch back to main
print(run("git checkout main"))

---

# Summary

You've seen:

1. **Multi-Agent System** (`app/graph.py`)
   - Supervisor pattern with invoice and music agents
   - LangGraph for orchestration

2. **Database Tools** (`app/tools.py`)
   - SQLAlchemy + Chinook database
   - Invoice and music catalog queries

3. **LangSmith Evaluation** (`tests/test_eval.py`)
   - Dataset-driven testing
   - LLM-as-judge for semantic correctness
   - 70% threshold for passing

4. **CI/CD Pipeline** (`.github/workflows/evaluate.yml`)
   - Triggers on agent code changes
   - Runs evals automatically
   - Posts results as PR comments

This pattern ensures your agent quality is validated on every change!